# Heatwave-AI — Data & Model Pipeline (Colab)

Runs the **real** pipeline: Open-Meteo (ERA5) → sWBGT → per-province percentile + persistence labels → temporal split → LightGBM (isotonic-calibrated) → writes forecasts + thresholds to **Supabase** and uploads the model to **Hugging Face**.

### Set these Colab secrets first (🔑 left sidebar → Secrets, toggle “Notebook access”)
| Secret | What | Where to get it |
|---|---|---|
| `DATABASE_URL` | Supabase **Session pooler** connection string | Supabase → Connect → *Session pooler* (IPv4, works from Colab). Looks like `postgresql://postgres.ejvrzprcbxgvaqbydagd:<PASSWORD>@aws-...pooler.supabase.com:5432/postgres` |
| `HF_TOKEN` | Hugging Face **write** token | huggingface.co → Settings → Access Tokens |
| `GITHUB_TOKEN` | *(only if the repo is private)* GitHub PAT with `repo` scope | github.com → Settings → Developer settings → PATs |

> Use the **Session pooler** string (port 5432), not the Transaction pooler (6543) — the Python writer (`psycopg`) uses statements the transaction pooler doesn’t support.

## 1. Clone the repo (branch `feat/region-line-oa`)

In [ ]:
REPO = 'https://github.com/MCTEEKUNG/Heatwave_Backend_Elysia.git'
BRANCH = 'feat/region-line-oa'

tok = None
try:
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

url = REPO.replace('https://', f'https://{tok}@') if tok else REPO
!rm -rf heatwave
!git clone --depth 1 -b $BRANCH $url heatwave
%cd heatwave

## 2. Install dependencies

In [ ]:
!pip -q install -r requirements.txt pyarrow 'psycopg[binary]' huggingface_hub

## 3. Load secrets into the environment

In [ ]:
import os
from google.colab import userdata
os.environ['DATABASE_URL'] = userdata.get('DATABASE_URL')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('DATABASE_URL set:', bool(os.environ.get('DATABASE_URL')))
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

## 4. Build the dataset (real Open-Meteo, 77 provinces × 1991–2025)
~77 archive calls; takes a few minutes. Writes `data/processed/dataset.parquet` + `province_thresholds.parquet`.

In [ ]:
!python -m pipeline.build_dataset

## 5. Train + calibrate + evaluate  —  🚦 QUALITY GATE
Read the printed metrics: **`test` PR-AUC / MCC / F2** must beat **`baseline_constant`**. If not, the model has no skill — add features (geopotential 500 hPa, soil moisture, more lags) before deploying.

In [ ]:
!python -m pipeline.train

## 6. Load thresholds into Supabase (`heatwave.province_thresholds`)

In [ ]:
!python -m pipeline.load_thresholds

## 7. Generate forecasts → Supabase (`heatwave.forecasts`)

In [ ]:
!python -m pipeline.run_forecast

## 8. Upload model + thresholds to Hugging Face (for Render to download in M4)

In [ ]:
from huggingface_hub import HfApi
REPO_ID = 'MCTEEKUNG123/Heatwave-AI'  # <-- change if your HF model repo differs
api = HfApi(token=os.environ['HF_TOKEN'])
api.upload_file(path_or_fileobj='models/heatwave_model.pkl',
                path_in_repo='models/heatwave_model.pkl', repo_id=REPO_ID)
api.upload_file(path_or_fileobj='data/processed/province_thresholds.parquet',
                path_in_repo='data/province_thresholds.parquet', repo_id=REPO_ID)
print('uploaded model + thresholds to', REPO_ID)

## Done ✅
- `heatwave.province_thresholds` + `heatwave.forecasts` populated in Supabase
- `heatwave_model.pkl` + thresholds parquet on Hugging Face

**Next:** verify the train metrics beat baseline (M2 gate), then M3 (LINE) → M4 (deploy: Render downloads the model from HF + daily cron runs `run_forecast`).